In [ ]:
# Import dependencies
import sys
import os
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

# Add api folder to path
api_path = Path(r'c:\Users\Amer\Desktop\Raytheon-Team-B-Group-7\api')
sys.path.insert(0, str(api_path))

print(f"API path: {api_path}")
print(f"Exists: {api_path.exists()}")

## Step 1: Load and Inspect the Model

In [ ]:
# Load model artifacts
model_dir = api_path / 'models'

feature_cols = joblib.load(model_dir / 'feature_cols.joblib')
scaler = joblib.load(model_dir / 'scaler.joblib')
rf_model = joblib.load(model_dir / 'random_forest.joblib')

print("Feature columns:", feature_cols)
print("\nNumber of features:", len(feature_cols))
print("\nModel type:", type(rf_model))

## Step 2: Test Model with Different Inputs

We'll create two clearly different feature vectors and verify the model returns different predictions.

In [ ]:
# Create two different feature vectors
features_1 = {
    'latitude': 35.0,
    'longitude': -120.0,
    'brightness': 320.0,
    'bright_t31': 290.0,
    'confidence': 80.0,
    'daynight': 1,
    'elevation': 500.0,
    'slope': 10.0,
    'aspect': 180.0,
    'temp': 30.0,
    'humidity': 20.0,
    'wind_speed': 15.0,
    'precip': 0.0,
    'month': 7
}

features_2 = {
    'latitude': 36.5,
    'longitude': -118.5,
    'brightness': 280.0,
    'bright_t31': 270.0,
    'confidence': 60.0,
    'daynight': 0,
    'elevation': 1500.0,
    'slope': 25.0,
    'aspect': 90.0,
    'temp': 22.0,
    'humidity': 45.0,
    'wind_speed': 8.0,
    'precip': 2.0,
    'month': 9
}

# Build feature vectors in correct order
def build_vector(features):
    return np.array([float(features[col]) for col in feature_cols]).reshape(1, -1)

X1 = build_vector(features_1)
X2 = build_vector(features_2)

print("Vector 1:", X1)
print("Vector 2:", X2)
print("\nAre they identical?", np.array_equal(X1, X2))

In [ ]:
# Get predictions
prob_1 = rf_model.predict_proba(X1)[0, 1]
prob_2 = rf_model.predict_proba(X2)[0, 1]

print(f"Prediction 1: {prob_1:.6f}")
print(f"Prediction 2: {prob_2:.6f}")
print(f"\nDifference: {abs(prob_1 - prob_2):.6f}")

if abs(prob_1 - prob_2) < 0.001:
    print("⚠️  WARNING: Predictions are nearly identical despite different inputs!")
    print("This suggests a model training issue.")
else:
    print("✅ Model produces different outputs for different inputs.")
    print("Bug is likely in feature construction or database writes.")

## Step 3: Simulate the Server's Feature Construction

Test the actual code path used in `predict_spread_animation`.

In [ ]:
# Simulate what happens in predict_spread_animation
import random

# Template (first cluster point)
template = {
    'latitude': 35.0,
    'longitude': -120.0,
    'brightness': 320.0,
    'bright_t31': 290.0,
    'confidence': 80.0,
    'daynight': 1,
    'elevation': 500.0,
    'slope': 10.0,
    'aspect': 180.0,
    'temp': 30.0,
    'humidity': 20.0,
    'wind_speed': 15.0,
    'precip': 0.0,
    'month': 7
}

FEATURE_KEYS = list(features_1.keys())

# Simulate building features for 3 different blocks
blocks_to_test = [
    {'block_id': 'CA-1000-2000', 'center_lat': 35.1, 'center_lon': -120.1, 'burning_neighbors': 1},
    {'block_id': 'CA-1001-2001', 'center_lat': 35.2, 'center_lon': -120.2, 'burning_neighbors': 2},
    {'block_id': 'CA-1002-2002', 'center_lat': 35.3, 'center_lon': -120.3, 'burning_neighbors': 3},
]

print("Testing feature construction for multiple blocks:\n")
predictions = []

for meta in blocks_to_test:
    # This is the CURRENT (BUGGY) implementation
    feat = {}
    for k in FEATURE_KEYS:
        if k == "latitude":
            feat["latitude"] = meta["center_lat"]
        elif k == "longitude":
            feat["longitude"] = meta["center_lon"]
        else:
            # BUG: Using template values without variation!
            feat[k] = template.get(k, 0)
    
    # Add jitter (current implementation)
    seed_base = f"{meta['block_id']}-0"
    rnd = random.Random(seed_base)
    feat["temp"] = float(feat.get("temp", 0)) + rnd.uniform(-2.0, 2.0)
    feat["humidity"] = float(feat.get("humidity", 0)) + rnd.uniform(-5.0, 5.0)
    feat["wind_speed"] = float(feat.get("wind_speed", 0)) + rnd.uniform(-1.5, 1.5)
    feat["slope"] = float(feat.get("slope", 0)) + rnd.uniform(-3.0, 3.0)
    feat["brightness"] = float(feat.get("brightness", 0)) + 8.0 * float(meta.get("burning_neighbors", 1))
    
    X = build_vector(feat)
    prob = rf_model.predict_proba(X)[0, 1]
    
    predictions.append(prob)
    print(f"Block {meta['block_id']}:")
    print(f"  Features: temp={feat['temp']:.1f}, humidity={feat['humidity']:.1f}, brightness={feat['brightness']:.1f}")
    print(f"  Prediction: {prob:.6f}\n")

# Check if predictions are too similar
prob_std = np.std(predictions)
print(f"Standard deviation of predictions: {prob_std:.6f}")

if prob_std < 0.01:
    print("\n⚠️  ISSUE FOUND: Predictions are too similar!")
    print("The jitter is not enough to create meaningful variation.")
    print("Need to use actual environmental data per location.")
else:
    print("\n✅ Predictions show reasonable variation.")

## Step 4: Check Database Values

Connect to the database and inspect actual stored values.

In [ ]:
import psycopg2

# Database connection
DB_CONFIG = {
    "host": "aws-1-us-east-2.pooler.supabase.com",
    "dbname": "postgres",
    "user": "postgres.ogzrpvdamptoiicxkzfg",
    "password": "firecast123!",
    "port": 5432,
}

try:
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    
    # Check fire_cell_state table
    cur.execute("""
        SELECT block_id, last_prob, instant_spread_probability, prob_sum, prob_count, updated_at
        FROM fire_cell_state
        ORDER BY updated_at DESC
        LIMIT 20
    """)
    
    rows = cur.fetchall()
    
    if rows:
        df = pd.DataFrame(rows, columns=['block_id', 'last_prob', 'instant_prob', 'prob_sum', 'prob_count', 'updated_at'])
        print("Recent predictions from fire_cell_state:\n")
        print(df)
        
        # Check for identical values
        unique_probs = df['instant_prob'].nunique()
        print(f"\nUnique probability values: {unique_probs}")
        
        if unique_probs == 1:
            print("⚠️  ISSUE CONFIRMED: All probabilities are identical in the database!")
        else:
            print("✅ Database contains varied probability values.")
    else:
        print("No data found in fire_cell_state table.")
    
    cur.close()
    conn.close()
    
except Exception as e:
    print(f"Error connecting to database: {e}")

## Step 5: Solution Summary

### Root Cause
The issue is in `predict_spread_animation()` line ~1080-1090:

```python
# Build feature dict: reuse template for non-location fields
feat = {}
for k in FEATURE_KEYS:
    if k == "latitude":
        feat["latitude"] = meta["center_lat"]
    elif k == "longitude":
        feat["longitude"] = meta["center_lon"]
    else:
        # BUG: Using template values for ALL blocks
        feat[k] = template.get(k, 0)
```

**Problem**: Every block gets the same `elevation`, `slope`, `aspect`, and weather values from the template.

The small jitter added later (`±2°C temp`, `±5% humidity`) is not enough to create meaningful variation in predictions.

### Solutions

1. **Query actual topographic data** per (lat, lon) from a GIS database
2. **Query weather data** per location from an API or database
3. **Increase jitter ranges** significantly (temporary fix)
4. **Use spatial interpolation** from known data points
5. **Load static datasets** (elevation, slope, aspect) into memory and lookup by coordinates